# IBM Data Analyst Capstone Project
## Technology Trends from the 2025 Stack Overflow Developer Survey

**Prepared by:** Aqib Hanif  
**Program:** IBM Data Analyst Professional Certificate

This project analyzes current and future programming-language, database, development-tool, and demographic trends using the published 2025 Stack Overflow Developer Survey results. The goal is to turn survey data into practical insights for learning, hiring, and technology decisions.

## Workflow
1. Data preparation
2. Data wrangling
3. Exploratory analysis
4. Visualization
5. Dashboard-ready outputs
6. Findings and implications

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("capstone_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 50)

## Data source
The analysis uses published 2025 Stack Overflow Developer Survey results. Values below are the rounded figures used in my final capstone presentation. Desired-use percentages represent future interest, not guaranteed forecasts.

In [ ]:
languages_current = pd.DataFrame({
    "Technology": ["PowerShell","C++","C#","Java","TypeScript","Bash/Shell","Python","SQL","HTML/CSS","JavaScript"],
    "Percent": [23.0,24.0,28.0,29.0,44.0,49.0,57.9,59.0,62.0,66.0]
})
languages_desired = pd.DataFrame({
    "Technology": ["C++","C#","Go","Bash/Shell","Rust","TypeScript","JavaScript","HTML/CSS","SQL","Python"],
    "Percent": [17.0,19.0,23.0,27.0,29.0,32.0,34.0,34.0,36.0,39.0]
})
databases_current = pd.DataFrame({
    "Technology": ["DynamoDB","Oracle","Elasticsearch","MariaDB","MongoDB","Redis","SQL Server","SQLite","MySQL","PostgreSQL"],
    "Percent": [10.0,11.0,17.0,22.0,24.0,28.0,30.0,37.5,40.5,55.7]
})
databases_desired = pd.DataFrame({
    "Technology": ["Supabase","DynamoDB","MariaDB","Elasticsearch","SQL Server","MongoDB","MySQL","Redis","SQLite","PostgreSQL"],
    "Percent": [6.0,7.0,13.0,13.0,15.0,18.0,21.0,23.5,28.0,46.5]
})
current_technology_usage = pd.DataFrame({
    "Technology": ["VS Code","Docker","Node.js","React"],
    "Percent": [75.9,71.1,48.7,44.7]
})
future_technology_trends = pd.DataFrame({
    "Technology": ["Docker","React","AWS","Kubernetes"],
    "Percent": [50.4,30.7,29.5,27.9]
})
demographics = pd.DataFrame({
    "Measure": ["Age 25-34","Age 35-44","Full-stack role","Student role"],
    "Percent": [33.6,26.9,27.0,11.3]
})

## Data wrangling
I combine the current and desired language/database tables, check duplicates and missing values, standardize labels, and create a normalized percentage field for comparison.

In [ ]:
lc = languages_current.assign(Category="Programming Language", Period="Current")
ld = languages_desired.assign(Category="Programming Language", Period="Desired")
dc = databases_current.assign(Category="Database", Period="Current")
dd = databases_desired.assign(Category="Database", Period="Desired")

technology_data = pd.concat([lc, ld, dc, dd], ignore_index=True)
print("Shape:", technology_data.shape)
print("Duplicates:", technology_data.duplicated().sum())
print("Missing values:\n", technology_data.isna().sum())

technology_clean = technology_data.drop_duplicates().copy()
technology_clean["Technology"] = technology_clean["Technology"].astype(str).str.strip()
min_v = technology_clean["Percent"].min()
max_v = technology_clean["Percent"].max()
technology_clean["Percent_Normalized"] = (technology_clean["Percent"] - min_v) / (max_v - min_v)
technology_clean = technology_clean.sort_values(["Category","Period","Percent"], ascending=[True,True,False]).reset_index(drop=True)
display(technology_clean.head(15))

## Exploratory analysis
I summarize the percentage distribution, check IQR outliers, and compare current versus desired percentages for technologies appearing in both lists.

In [ ]:
display(technology_clean["Percent"].describe().to_frame("Percent"))

q1 = technology_clean["Percent"].quantile(0.25)
q3 = technology_clean["Percent"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = technology_clean[(technology_clean["Percent"] < lower) | (technology_clean["Percent"] > upper)]
print(f"IQR bounds: {lower:.2f} to {upper:.2f}")
display(outliers)

def overlap(current_df, desired_df):
    return current_df.rename(columns={"Percent":"Current"}).merge(
        desired_df.rename(columns={"Percent":"Desired"}), on="Technology", how="inner"
    )

lang_overlap = overlap(languages_current, languages_desired)
db_overlap = overlap(databases_current, databases_desired)
print("Language correlation:", round(lang_overlap["Current"].corr(lang_overlap["Desired"]), 3))
print("Database correlation:", round(db_overlap["Current"].corr(db_overlap["Desired"]), 3))

## Programming language trends
JavaScript has the highest current usage in the project snapshot, while Python leads the desired-use list. SQL remains strong in both current and desired lists.

In [ ]:
for data, title, filename in [
    (languages_current, "Current Year: Programming Languages Used", "languages_current.png"),
    (languages_desired, "Next Year: Programming Languages Desired", "languages_desired.png")
]:
    plot_df = data.sort_values("Percent")
    plt.figure(figsize=(10,6))
    bars = plt.bar(plot_df["Technology"], plot_df["Percent"])
    for bar, value in zip(bars, plot_df["Percent"]):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f"{value:.1f}", ha="center")
    plt.title(title)
    plt.ylabel("Respondents (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=150)
    plt.show()

## Database trends
PostgreSQL is the strongest database signal in both current use and desired interest. MySQL, SQLite, Redis, and MongoDB also remain relevant for practical data and application workflows.

In [ ]:
for data, title, filename in [
    (databases_current, "Current Year: Databases Used", "databases_current.png"),
    (databases_desired, "Next Year: Databases Desired", "databases_desired.png")
]:
    plot_df = data.sort_values("Percent")
    plt.figure(figsize=(10,6))
    bars = plt.bar(plot_df["Technology"], plot_df["Percent"])
    for bar, value in zip(bars, plot_df["Percent"]):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f"{value:.1f}", ha="center")
    plt.title(title)
    plt.ylabel("Respondents (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=150)
    plt.show()

## Dashboard-ready summary
Current technology usage highlights VS Code and Docker. Future-interest data highlights Docker, React, AWS, and Kubernetes. The demographic snapshot shows the largest selected age group is 25-34.

In [ ]:
dashboard_summary = pd.concat([
    current_technology_usage.rename(columns={"Technology":"Measure"}).assign(Section="Current Technology Usage"),
    future_technology_trends.rename(columns={"Technology":"Measure"}).assign(Section="Future Technology Trends"),
    demographics.assign(Section="Demographics")
], ignore_index=True)[["Section","Measure","Percent"]]
display(dashboard_summary)

technology_clean.to_csv(OUTPUT_DIR / "technology_trends.csv", index=False)
dashboard_summary.to_csv(OUTPUT_DIR / "dashboard_summary.csv", index=False)

## Key findings and implications
- **Python + SQL** form the strongest foundation for a data-focused learning path.
- **PostgreSQL** is the leading database signal in both current and desired use.
- **Docker, AWS, and Kubernetes** show the importance of cloud-native and reproducible workflows.
- **JavaScript, TypeScript, React, and Node.js** remain useful for modern data products and web-based delivery.
- The best portfolio projects connect data collection, cleaning, analysis, visualization, dashboards, and business interpretation.

## Conclusion
This capstone shows that a strong data analyst should combine technical capability with clear communication. My recommended stack is **Python + SQL + PostgreSQL + visualization + Git/GitHub + APIs + cloud awareness**.

**Prepared by: Aqib Hanif**